<a href="https://colab.research.google.com/github/juanpa2191/audio-texto-hugging-face/blob/main/audio_a_texto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai-whisper pyannote.audio

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.68.1 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.68.1 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.68.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


In [ ]:
import whisper, torch
from pyannote.audio import Pipeline

HF_TOKEN = userdata.get("HF_TOKEN")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model("large-v3", device=device)

try:  # pyannote 4.x
    diarizer = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1", token=HF_TOKEN)
except TypeError:  # pyannote 3.x
    diarizer = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1", use_auth_token=HF_TOKEN)

if diarizer is None:
    raise RuntimeError("No se pudo cargar pyannote: revisa el token y las condiciones del modelo.")
diarizer.to(torch.device(device))
print("Modelos listos ✅ en", device)

100%|█████████████████████████████████████| 2.88G/2.88G [02:23<00:00, 21.5MiB/s]


config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 5.91MB            

segmentation/pytorch_model.bin: downloading bytes:           |  0.00B            

plda/xvec_transform.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/xvec_transform.npz: downloading bytes:           |  0.00B            

plda/plda.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/plda.npz: downloading bytes:           |  0.00B            

embedding/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 26.6MB            

embedding/pytorch_model.bin: downloading bytes:           |  0.00B            

Modelos listos ✅ en cuda


In [ ]:
import os, shutil, time
from google.colab import files

IDIOMA = "es"
NUM_HABLANTES = None   # ej. 2 si sabes cuántos hablan
NOMBRES = {}           # ej. {"Hablante 1": "Juan", "Hablante 2": "Cliente"}

os.makedirs("audios", exist_ok=True)
os.makedirs("transcripciones", exist_ok=True)

def fmt(t):
    h, r = divmod(int(t), 3600); m, s = divmod(r, 60)
    return f"{h:02d}:{m:02d}:{s:02d}" if h else f"{m:02d}:{s:02d}"

def diarizar(wav):
    data = {"waveform": torch.from_numpy(wav).unsqueeze(0), "sample_rate": 16000}
    kw = {"num_speakers": NUM_HABLANTES} if NUM_HABLANTES else {}
    out = diarizer(data, **kw)
    ann = getattr(out, "speaker_diarization", out)
    return [(t.start, t.end, spk) for t, _, spk in ann.itertracks(yield_label=True)]

def quien_habla(seg, turnos):
    mejor, solape = None, 0
    for a, b, spk in turnos:
        o = min(seg["end"], b) - max(seg["start"], a)
        if o > solape:
            mejor, solape = spk, o
    return mejor or "?"

subidos = files.upload()
salidas = []

for nombre in subidos:
    ruta = os.path.join("audios", nombre)
    shutil.move(nombre, ruta)
    print(f"\n🎧 {nombre}")
    inicio = time.time()

    wav = whisper.load_audio(ruta)
    print("  · Transcribiendo...")
    r = model.transcribe(wav, language=IDIOMA, fp16=(device == "cuda"))
    print("  · Identificando voces...")
    turnos = diarizar(wav)

    etiquetas = {}
    for _, _, spk in sorted(turnos):
        etiquetas.setdefault(spk, f"Hablante {len(etiquetas) + 1}")

    bloques = []
    for s in r["segments"]:
        spk = etiquetas.get(quien_habla(s, turnos), "?")
        spk = NOMBRES.get(spk, spk)
        if bloques and bloques[-1]["spk"] == spk:
            bloques[-1]["texto"] += " " + s["text"].strip()
        else:
            bloques.append({"inicio": s["start"], "spk": spk, "texto": s["text"].strip()})

    salida = os.path.join("transcripciones", os.path.splitext(nombre)[0] + ".txt")
    with open(salida, "w", encoding="utf-8") as f:
        for b in bloques:
            f.write(f"[{fmt(b['inicio'])}] {b['spk']}: {b['texto']}\n\n")

    salidas.append(salida)
    print(f"✅ {len(etiquetas)} voces · {time.time() - inicio:.0f}s → {salida}")

if len(salidas) == 1:
    files.download(salidas[0])
elif salidas:
    tmp = f"/tmp/transcripciones_{time.strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(tmp, exist_ok=True)
    for s in salidas: shutil.copy(s, tmp)
    shutil.make_archive(tmp, "zip", tmp)
    files.download(tmp + ".zip")

Saving Wednesday, 3h42m PM.m4a to Wednesday, 3h42m PM.m4a

🎧 Wednesday, 3h42m PM.m4a
  · Transcribiendo...
  · Identificando voces...


/usr/local/lib/python3.13/dist-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/usr/local/lib/python3.13/dist-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1913.)
  std = sequences.std(dim=-1, correction=1)


✅ 3 voces · 797s → transcripciones/Wednesday, 3h42m PM.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>